<a href="https://colab.research.google.com/github/humaaslam46/Internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humaaslam46/Internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content item (content_hash_id), for one client (client_hash_id), on one
report_date. My lane (Refresh/Content Opportunity Scoring) uses the mid-panel month
2026-03 to build and test features - never the _sample table, since that's the sealed
final month (June 2026) and would leak the natural outcome window into any label I build.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print("Connected.")

KeyboardInterrupt: Interrupted by user

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

Feature: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions (first half of March
only - the "knowable" window).
Label/proxy: is_declining - whether the second half of March's impressions fell more than
20% versus the first half, built only from data inside March itself.
Context: client_hash_id, content_hash_id - grouping/joins only, never features.
Excluded (deliberately): trend_direction/trend_pct-style pre-computed labels, and anything
from April onward - that's the actual future outcome window my label is a proxy for.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# --- Query 1: grain check ---
grain = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS n_distinct_keys
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print("Grain check (should match):"); print(grain)

# --- Query 2: row count + date span ---
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS start_date, MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print("\nRow count + date span:"); print(span)

# --- Query 3: availability, filtered with IS TRUE ---
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print("\nAvailability (IS TRUE filter):"); print(avail)

# --- Five features, from the FIRST half of March only (knowable at decision moment) ---
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_first_half,
           SUM(gsc_clicks) AS clicks_first_half,
           AVG(gsc_avg_position) AS avg_position_first_half,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days_first_half,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions_first_half
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND report_date <= DATE '2026-03-15'
    GROUP BY 1, 2
""").df()
print(f"\n{len(feat):,} content items with first-half-of-March features")

# Second half - used ONLY to build the label, never as a feature
label_half = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND report_date > DATE '2026-03-15'
    GROUP BY 1, 2
""").df()

data = feat.merge(label_half, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["imp_second_half"] < 0.8 * data["imp_first_half"]).astype(int)

# --- The trap: add the label-derived column on purpose ---
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

honest_features = ["imp_first_half", "clicks_first_half", "avg_position_first_half",
                    "active_days_first_half", "ga4_sessions_first_half"]

X = data[honest_features].fillna(0)
y = data["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print("\nHonest score (5 features only):", round(honest_model.score(X_te, y_te), 3))

# now leak it
X_leaky = data[honest_features + ["imp_second_half"]].fillna(0)  # <- this IS the label, in disguise
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2)
print("Leaky score (with imp_second_half added):", round(leaky_model.score(X_te2, y_te2), 3), " <- jumps toward perfect")

# delete it, keep the honest number
print("\nKept: the honest 5-feature score above. imp_second_half is deleted, not used.")

## 4. Data limits

This slice can never tell me *why* a page's impressions dropped - only that they did.
It also can't generalize across the full panel: client history depth is unbalanced
(dim_clients.gsc_data_start varies by client), so a pattern found in March 2026 may not
hold for a client that joined the platform six months later.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.